# NB-01-2 ブロッホ球：量子ビットの状態を球面上の点として見る

[パウリ行列のノート](../../研究ノート/04_群論・代数/pauli_matrices_derivation_and_group.md#p6)の Part VI（$\hat n\cdot\vec\sigma$ の固有ベクトルとブロッホ球）を、数値計算とスライダーで確かめます。Qiskit の `Statevector` と `DensityMatrix` でも答え合わせをします（量子回路は使いません。回路は Phase 2 の NB-07 で扱います）。

**確かめること**

1. $|{+}\hat n\rangle=\cos\frac\theta2|0\rangle+e^{i\phi}\sin\frac\theta2|1\rangle$ は $\hat n\cdot\vec\sigma$ の固有値 $+1$ の固有ベクトルで、どの状態も（大域位相を除いて）この形になる（Part VI §2・§3）
2. パウリ行列の期待値は $\hat n$、射影は $\frac12(I+\hat n\cdot\vec\sigma)$（§4）
3. 重なり $|\langle{+}\hat m|{+}\hat n\rangle|^2=\frac12(1+\hat m\cdot\hat n)$、つまり直交する状態は対蹠点（§5）
4. 測定の確率 $P(0)=\cos^2\frac\theta2$ を、Qiskit のサンプリングで確かめる
5. 混合状態はブロッホ球の内側にあり、純度は $\frac12(1+|\vec r|^2)$（§6）

**使い方**：上から順に実行します。3次元の図は、マウスのドラッグで回転、ホイールで拡大できます。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from numpy.typing import NDArray
from plotly.basedatatypes import BaseTraceType
from qiskit.quantum_info import DensityMatrix, Pauli, Statevector

# 型の別名（型ヒントを読みやすくするため。実行時には何も確かめない）
Matrix = NDArray[np.complex128]   # 2×2 の複素行列（演算子・密度行列）
Ket = NDArray[np.complex128]      # 長さ 2 の複素ベクトル（状態ベクトル）
Vector = NDArray[np.float64]      # 3次元の実ベクトル（ブロッホベクトル）

sx: Matrix = np.array([[0, 1], [1, 0]], dtype=complex)
sy: Matrix = np.array([[0, -1j], [1j, 0]])
sz: Matrix = np.array([[1, 0], [0, -1]], dtype=complex)
S: tuple[Matrix, Matrix, Matrix] = (sx, sy, sz)
I2: Matrix = np.eye(2, dtype=complex)


def nvec(theta: float, phi: float) -> Vector:
    """球座標の単位ベクトル（θ：北極からの角度、φ：経度。ラジアン）"""
    return np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])


def ket(theta: float, phi: float) -> Ket:
    """ノートの Part VI §2：n·σ の固有値 +1 の固有ベクトル"""
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])


def bloch(psi_or_rho: Ket | Matrix) -> Vector:
    """ブロッホベクトル（⟨σx⟩, ⟨σy⟩, ⟨σz⟩）。状態ベクトルでも密度行列でもよい"""
    rho = psi_or_rho if np.ndim(psi_or_rho) == 2 else np.outer(psi_or_rho, psi_or_rho.conj())
    return np.array([np.trace(rho @ Sk).real for Sk in S])


def angles(psi: Ket) -> tuple[float, float]:
    """大域位相をそろえて（α を 0 以上の実数にして）、(θ, φ) を求める（Part VI §3）"""
    psi = psi / np.linalg.norm(psi)
    psi = psi * np.exp(-1j * np.angle(psi[0])) if abs(psi[0]) > 1e-12 else psi * np.exp(-1j * np.angle(psi[1]))
    return 2 * np.arccos(np.clip(psi[0].real, -1, 1)), np.angle(psi[1])

## 1. 数値で確認する

ランダムな状態で、Part VI の式をまとめて確かめます。Qiskit の `Statevector.expectation_value` が返す期待値も、自前の計算と比べます。

In [ ]:
rng = np.random.default_rng(0)
for _ in range(200):
    th, ph = rng.uniform(0, np.pi), rng.uniform(-np.pi, np.pi)
    n, v = nvec(th, ph), ket(th, ph)
    N = n[0] * sx + n[1] * sy + n[2] * sz
    assert np.allclose(N @ v, v)                                                    # 固有値 +1 の固有ベクトル
    assert np.allclose(bloch(v), n)                                                 # ⟨σ⟩ = n
    assert np.allclose(np.outer(v, v.conj()), 0.5 * (I2 + N))                       # 射影 ½(I + n·σ)
    sv = Statevector(v)
    assert np.allclose([sv.expectation_value(Pauli(p)).real for p in "XYZ"], n)     # Qiskit とも一致
    # どの状態も、大域位相を除いて ket(θ, φ)
    psi = rng.normal(size=2) + 1j * rng.normal(size=2)
    t2, p2 = angles(psi)
    assert np.isclose(abs(np.vdot(ket(t2, p2), psi / np.linalg.norm(psi))), 1)
    # 重なり ½(1 + m·n)、軸 m で測って +1 が出る確率も同じ式
    m = nvec(t2, p2)
    assert np.isclose(abs(np.vdot(ket(t2, p2), v)) ** 2, 0.5 * (1 + m @ n))
    Pm = 0.5 * (I2 + m[0] * sx + m[1] * sy + m[2] * sz)
    assert np.isclose(np.vdot(v, Pm @ v).real, 0.5 * (1 + m @ n))
print("Part VI の式を、ランダムな 200 個の状態で確認しました")
print("例：θ = 60°, φ = 45° の状態", np.round(ket(np.pi / 3, np.pi / 4), 4), "のブロッホベクトル", np.round(bloch(ket(np.pi / 3, np.pi / 4)), 4))

## 2. スライダーで状態を動かす

- **θ, φ**：状態 $\cos\frac\theta2|0\rangle+e^{i\phi}\sin\frac\theta2|1\rangle$ を決めます。赤い矢印がブロッホベクトル $\hat n$、灰色の点が直交する状態（対蹠点）です。
- **大域位相 γ**：状態全体に $e^{i\gamma}$ を掛けます。振幅（下の表示）は変わりますが、ブロッホ球の点は動きません。大域位相は観測できない、ということです（Part VI §3）。

In [ ]:
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなく Python のリストを渡す。
# numpy 配列を渡すと、ブラウザから戻ってくる差分の処理で
# 「ValueError: The truth value of an array ... is ambiguous」になることがある（plotly 7.1 の FigureWidget）。
def sphere_traces() -> list[BaseTraceType]:
    u, w = np.meshgrid(np.linspace(0, np.pi, 30), np.linspace(0, 2 * np.pi, 60))
    tr = [go.Surface(x=(np.sin(u) * np.cos(w)).tolist(), y=(np.sin(u) * np.sin(w)).tolist(), z=np.cos(u).tolist(), opacity=0.12, showscale=False,
                     colorscale=[[0, "#9ecae1"], [1, "#9ecae1"]], hoverinfo="skip")]
    t = np.linspace(0, 2 * np.pi, 200)
    tr.append(go.Scatter3d(x=np.cos(t).tolist(), y=np.sin(t).tolist(), z=(0 * t).tolist(), mode="lines", line=dict(color="#bbb"), hoverinfo="skip", showlegend=False))
    for e in np.eye(3):
        tr.append(go.Scatter3d(x=[-1.25 * e[0], 1.25 * e[0]], y=[-1.25 * e[1], 1.25 * e[1]], z=[-1.25 * e[2], 1.25 * e[2]],
                               mode="lines", line=dict(color="#ccc"), hoverinfo="skip", showlegend=False))
    names = {(0, 0, 1): "|0⟩", (0, 0, -1): "|1⟩", (1, 0, 0): "|+⟩", (-1, 0, 0): "|−⟩", (0, 1, 0): "|+i⟩", (0, -1, 0): "|−i⟩"}
    pts = np.array(list(names))
    tr.append(go.Scatter3d(x=(1.12 * pts[:, 0]).tolist(), y=(1.12 * pts[:, 1]).tolist(), z=(1.12 * pts[:, 2]).tolist(), mode="text", text=list(names.values()),
                           textfont=dict(size=14, color="#1f77b4"), hoverinfo="skip", showlegend=False))
    return tr


def scene_layout() -> dict[str, object]:
    ax = dict(range=[-1.3, 1.3], showbackground=False, showticklabels=False, title="")
    return dict(xaxis=ax, yaxis=ax, zaxis=ax, aspectmode="cube", camera=dict(eye=dict(x=1.3, y=-1.5, z=0.6)))


fig = go.FigureWidget(data=sphere_traces())
fig.add_trace(go.Scatter3d(x=[0, 0], y=[0, 0], z=[0, 1], mode="lines+markers", line=dict(color="#d62728", width=8),
                           marker=dict(size=[0, 6], color="#d62728"), name="ブロッホベクトル n"))
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[-1], mode="markers", marker=dict(size=6, color="#777"), name="直交する状態（−n）"))
fig.add_trace(go.Scatter3d(x=[0, 0, 0], y=[0, 0, 0], z=[0, 0, 0], mode="lines", line=dict(color="#d62728", dash="dot", width=3),
                           showlegend=False, hoverinfo="skip"))
fig.update_layout(height=560, width=640, margin=dict(l=0, r=0, t=10, b=0), scene=scene_layout(), legend=dict(y=0.02))

theta = W.FloatSlider(value=60, min=0, max=180, step=1, description="θ [度]")
phi = W.FloatSlider(value=45, min=-180, max=180, step=1, description="φ [度]")
gamma = W.FloatSlider(value=0, min=0, max=360, step=5, description="大域位相 γ")
info = W.HTML()


def fmt(z: complex) -> str:
    return f"{z.real:+.3f}{z.imag:+.3f}i"


def update(*_: object) -> None:
    """スライダーが動くたびに呼ばれ、図と文字の表示を描き直す"""
    th, ph, ga = np.radians(theta.value), np.radians(phi.value), np.radians(gamma.value)
    psi = np.exp(1j * ga) * ket(th, ph)
    n = bloch(psi)
    assert np.allclose(n, nvec(th, ph))                                     # 大域位相によらない
    with fig.batch_update():
        arrow, anti, proj = fig.data[-3], fig.data[-2], fig.data[-1]
        arrow.x, arrow.y, arrow.z = [0, n[0]], [0, n[1]], [0, n[2]]
        anti.x, anti.y, anti.z = [-n[0]], [-n[1]], [-n[2]]
        proj.x, proj.y, proj.z = [0, n[0], n[0]], [0, n[1], n[1]], [0, 0, n[2]]
    p0 = abs(psi[0]) ** 2
    info.value = (f"<b>状態</b> α|0⟩ + β|1⟩：α = {fmt(psi[0])}、β = {fmt(psi[1])}<br>"
                  f"<b>測定の確率</b> P(0) = |α|² = {p0:.3f}（= cos²(θ/2)）、P(1) = {1 - p0:.3f}<br>"
                  f"<b>期待値</b> (⟨σx⟩, ⟨σy⟩, ⟨σz⟩) = ({n[0]:+.3f}, {n[1]:+.3f}, {n[2]:+.3f}) = n")


for s in (theta, phi, gamma):
    s.observe(update, "value")
update()
W.VBox([W.HBox([theta, phi, gamma]), info, fig])

## 3. 測定をサンプリングで確かめる

状態 $|{+}\hat n\rangle$ で $\sigma_z$ を測ると、$|0\rangle$（$+1$）が出る確率は $\cos^2\frac\theta2$ です（Part VI §5）。Qiskit の `Statevector.sample_counts` で、各 $\theta$ について 400 回ずつ測定をサンプリングし、出た割合を理論の曲線と比べます（量子回路は使わず、状態ベクトルから直接サンプリングします）。

In [ ]:
shots = 400
thetas: Vector = np.radians(np.arange(0, 181, 10))
freq: list[float] = []
for k, th in enumerate(thetas):
    sv = Statevector(ket(th, 0.3))
    sv.seed(100 + k)                                   # 再現できるように乱数の種を固定
    counts = sv.sample_counts(shots)
    freq.append(counts.get("0", 0) / shots)
    assert np.isclose(sv.probabilities()[0], np.cos(th / 2) ** 2)
freq = np.array(freq)
assert np.all(np.abs(freq - np.cos(thetas / 2) ** 2) < 4 * np.sqrt(0.25 / shots))     # 統計誤差（4σ）の範囲

tt = np.linspace(0, np.pi, 200)
fig_m = go.Figure([go.Scatter(x=np.degrees(tt), y=np.cos(tt / 2) ** 2, mode="lines", name="理論 cos²(θ/2)"),
                   go.Scatter(x=np.degrees(thetas), y=freq, mode="markers", marker=dict(size=9, color="#d62728"),
                              name=f"サンプリング（{shots} 回ずつ）")])
fig_m.update_layout(height=380, width=700, xaxis_title="θ [度]", yaxis_title="|0⟩ が出た割合", margin=dict(t=20))
fig_m

## 4. 混合状態：ブロッホ球の内側

2つの純粋状態 $\hat n_1$（北極 $|0\rangle$）と $\hat n_2$ を、確率 $p$ と $1-p$ で混ぜた密度行列 $\rho=p|\psi_1\rangle\langle\psi_1|+(1-p)|\psi_2\rangle\langle\psi_2|$ のブロッホベクトルは、$\vec r=p\,\hat n_1+(1-p)\,\hat n_2$ です（2点を結ぶ線分の上）。

- **p**：混ぜる割合
- **Θ**：$\hat n_1$ と $\hat n_2$ のなす角。$\Theta=180^\circ$（直交する状態）で $p=0.5$ にすると、球の中心（完全な混合状態 $\rho=I/2$）になります。

純度は Qiskit の `DensityMatrix.purity()` でも計算して、$\frac12(1+|\vec r|^2)$ と比べます。

In [ ]:
fig_r = go.FigureWidget(data=sphere_traces())
fig_r.add_trace(go.Scatter3d(x=[0, 0], y=[0, 0], z=[1, -1], mode="lines+markers", line=dict(color="#e67e00", dash="dash", width=4),
                             marker=dict(size=5, color="#e67e00"), name="混ぜる2つの純粋状態"))
fig_r.add_trace(go.Scatter3d(x=[0, 0], y=[0, 0], z=[0, 0.4], mode="lines+markers", line=dict(color="#d62728", width=8),
                             marker=dict(size=[0, 7], color="#d62728"), name="ブロッホベクトル r"))
fig_r.update_layout(height=520, width=600, margin=dict(l=0, r=0, t=10, b=0), scene=scene_layout(), legend=dict(y=0.02))

p_slider = W.FloatSlider(value=0.7, min=0, max=1, step=0.05, description="p")
Th_slider = W.FloatSlider(value=180, min=0, max=180, step=5, description="Θ [度]")
info_r = W.HTML()


def update_r(*_: object) -> None:
    """スライダーが動くたびに呼ばれ、図と文字の表示を描き直す"""
    p, Th = p_slider.value, np.radians(Th_slider.value)
    n1, n2 = nvec(0, 0), nvec(Th, 0)
    v1, v2 = ket(0, 0), ket(Th, 0)
    rho = p * np.outer(v1, v1.conj()) + (1 - p) * np.outer(v2, v2.conj())
    r = bloch(rho)
    assert np.allclose(r, p * n1 + (1 - p) * n2)
    R = np.linalg.norm(r)
    purity = np.trace(rho @ rho).real
    assert np.isclose(purity, 0.5 * (1 + R**2)) and np.isclose(DensityMatrix(rho).purity().real, purity)
    lam = np.linalg.eigvalsh(rho)
    assert np.allclose(sorted(lam), [(1 - R) / 2, (1 + R) / 2])
    with fig_r.batch_update():
        pair, arrow = fig_r.data[-2], fig_r.data[-1]
        pair.x, pair.y, pair.z = [n1[0], n2[0]], [n1[1], n2[1]], [n1[2], n2[2]]
        arrow.x, arrow.y, arrow.z = [0, r[0]], [0, r[1]], [0, r[2]]
    info_r.value = (f"|r| = {R:.3f}、純度 tr ρ² = {purity:.3f}（= (1+|r|²)/2、Qiskit の purity() とも一致）、"
                    f"固有値 = ({lam[0]:.3f}, {lam[1]:.3f}) = (1 ∓ |r|)/2")


for s in (p_slider, Th_slider):
    s.observe(update_r, "value")
update_r()
W.VBox([W.HBox([p_slider, Th_slider]), info_r, fig_r])

## 5. Qiskit の描画と比べる

θ = 60°、φ = 90° の状態を、Qiskit の `plot_bloch_multivector`（状態ベクトルから）と `plot_bloch_vector`（ブロッホベクトルから）で描きます。上のスライダーの図で θ = 60°、φ = 90° にしたときと同じ点になっていることを確かめてください（Qiskit の図は、$x$ 軸が手前、$y$ 軸が右向きです）。

In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_bloch_multivector, plot_bloch_vector

plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "sans-serif"]   # 図のタイトルの日本語用（Windows のフォント）

th, ph = np.radians(60), np.radians(90)
sv = Statevector(ket(th, ph))
display(plot_bloch_multivector(sv, title="θ = 60°, φ = 90°（状態ベクトルから）"))
display(plot_bloch_vector(list(bloch(ket(th, ph))), title="同じ点（ブロッホベクトルから）"))

## 観察のポイント

- θ を 0 から 180° まで動かすと、状態は $|0\rangle$ から $|1\rangle$ まで変わり、$P(0)=\cos^2\frac\theta2$ が 1 から 0 まで減ります。ブロッホ球の角度 θ に対して、状態の振幅には半分の角度 θ/2 が現れます（Part VI §5）。
- 北極・南極では、φ を動かしても点が動きません。極は座標特異点です（Part VI §3 の補足）。
- 大域位相 γ を動かしても、ブロッホ球の点も測定の確率も変わりません。
- 混合状態のブロッホベクトルは、混ぜた純粋状態を結ぶ線分の上にあり、球の内側に入ります。純粋状態だけが球面上にあります。

**次に試すこと**：混合状態のセルで、Θ = 90°（たとえば $|0\rangle$ と $|{+}\rangle$）にすると、$p$ をどう選んでも球の中心には届きません。中心に届くのは、どんな2状態を混ぜたときでしょうか。

**次のノートブック（NB-02）**：[../02_su2_rotation/01_su2_exponential.ipynb](../02_su2_rotation/01_su2_exponential.ipynb) と [02_bloch_rotation.ipynb](../02_su2_rotation/02_bloch_rotation.ipynb) で、$e^{-i\theta\,\hat n\cdot\vec\sigma/2}$ でブロッホベクトルが回る様子（ノートの Part VII）を扱います。